# 🔧 Fine-Tuning Transformers (DistilBERT)

In the previous section on Transfer Learning (CV), we manually froze layers and replaced the classification head of a ResNet.

In the world of **Natural Language Processing (NLP)**, the concept is identical, but the tools and architecture are slightly different. The dominant paradigm is **"Pre-train, then Fine-tune."**

### The Concept
1.  **Pre-training (The "Language" Phase):** A model (like BERT) is trained on massive amounts of text (Wikipedia, BookCorpus) using self-supervised objectives (Masked Language Modeling). It learns syntax, grammar, and world knowledge.
2.  **Fine-tuning (The "Task" Phase):** We take this "smart" language model, add a task-specific layer on top, and train it on a smaller, labeled dataset (e.g., sentiment analysis, spam detection).



### In this notebook, we will:
1.  **Load a Pre-trained Model:** We will use **DistilBERT**, a lighter, faster version of BERT that retains 97% of performance.
2.  **Prepare Text Data:** Use the Hugging Face `datasets` library to load and tokenize the IMDb movie review dataset.
3.  **The `Trainer` API:** Use the high-level `Trainer` class, which handles the training loop, logging, and evaluation automatically.
4.  **Inference:** Test our fine-tuned model on new, unseen sentences.

## 1. Setup and Installation

We need the standard Hugging Face stack.
* `transformers`: The model architectures.
* `datasets`: To easily load text data.
* `evaluate`: To calculate metrics like accuracy.
* `accelerate`: Optimizes the training loop (required by `Trainer`).

In [1]:
!pip install -q transformers datasets evaluate accelerate

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
Using device: cuda


## 2. Data Preparation: The IMDb Dataset

We will use the **IMDb dataset**, a classic binary classification task:
* **Input:** A movie review text.
* **Label:** 0 (Negative) or 1 (Positive).

We will create a smaller subset of the data to ensure this notebook runs quickly.

In [3]:
# 1. Load Dataset
# The dataset has 'train' and 'test' splits
dataset = load_dataset("imdb")

# 2. Create a smaller subset for faster training demonstration
# We'll use 2000 samples for training and 500 for validation
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(f"Sample review: {small_train_dataset[0]['text'][:200]}...")
print(f"Label: {small_train_dataset[0]['label']} (0=Neg, 1=Pos)")

Sample review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...
Label: 1 (0=Neg, 1=Pos)


## 3. Tokenization

Transformers cannot read raw text. We must convert text into numbers (Token IDs).

We use `AutoTokenizer`. Crucially, we must use the **exact same tokenizer** that was used during the model's pre-training.

**Key operations:**
* **Padding:** Making all sequences in a batch the same length.
* **Truncation:** Cutting off sequences that are longer than the model's maximum limit (usually 512 tokens).

In [5]:
# 1. Load the tokenizer associated with DistilBERT
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# 2. Define a processing function
def tokenize_function(examples):
    # Padding="max_length" ensures all vectors are size 512
    # Truncation=True cuts off reviews longer than 512 tokens
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# 3. Apply to the whole dataset efficiently using .map()
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_eval = small_eval_dataset.map(tokenize_function, batched=True)

# Look at the result features
print(tokenized_train[0].keys())

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'input_ids', 'attention_mask'])


## 4. Model Setup: `AutoModelForSequenceClassification`

When we load the pre-trained model with `AutoModelForSequenceClassification`, the library automatically:
1.  **Loads the Body:** Loads the pre-trained DistilBERT layers (weights trained on English).
2.  **Replaces the Head:** Discards the pre-training head and replaces it with a **classification head** (a linear layer with `num_labels` outputs), initialized with random weights.

In [6]:
# num_labels=2 because we have Positive vs Negative
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Move model to device
model.to(device)

print("Model loaded. Pre-trained body + Randomly initialized Classification Head.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded. Pre-trained body + Randomly initialized Classification Head.


## 5. The `Trainer` API

We use the Hugging Face `Trainer` class, which handles the complex, optimized training loop, logging, and evaluation automatically.

In [9]:
# 1. Define Evaluation Metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # The model outputs logits (raw scores) -> take argmax to get class 0 or 1
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",          # Where to store checkpoints
    eval_strategy="epoch",           # Evaluate at the end of every epoch
    save_strategy="epoch",           # Save model at the end of every epoch
    learning_rate=2e-5,              # Low learning rate is critical for Fine-Tuning!
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,              # 2 epochs is usually enough for this task
    weight_decay=0.01,               # Regularization
    logging_steps=50,                # Log every 50 steps
    report_to="none"                 # Disable W&B logging for this demo
)

# 3. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

## 6. Training and Evaluation

We call `trainer.train()` to begin the fine-tuning process.

In [10]:
print("Starting Training...")
trainer.train()

# Final Evaluation
results = trainer.evaluate()
print(f"\nFinal Test Accuracy: {results['eval_accuracy']:.4f}")

Starting Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.413500,0.313895,0.860000
2,0.205900,0.303338,0.874000



Final Test Accuracy: 0.8740


## 7. Inference

We use the `pipeline` API, passing our newly fine-tuned model to easily classify new text inputs.

In [11]:
from transformers import pipeline

# Load the pipeline using OUR fine-tuned model and tokenizer
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Test examples
test_reviews = [
    "This movie was an absolute masterpiece! The acting was superb.",
    "I fell asleep halfway through. Complete waste of time.",
    "It was okay, not great but not terrible either."
]

predictions = classifier(test_reviews)

for review, pred in zip(test_reviews, predictions):
    print(f"Review: {review}")
    print(f"Prediction: {pred['label']} (Score: {pred['score']:.4f})\n")

Device set to use cuda:0


Review: This movie was an absolute masterpiece! The acting was superb.
Prediction: LABEL_1 (Score: 0.9595)

Review: I fell asleep halfway through. Complete waste of time.
Prediction: LABEL_0 (Score: 0.9599)

Review: It was okay, not great but not terrible either.
Prediction: LABEL_1 (Score: 0.5388)



## 8. Conclusion

We have successfully fine-tuned a Transformer!

**Key Takeaways:**
1.  **Architecture:** We took a pre-trained **DistilBERT** and automatically added a classification head.
2.  **Efficiency:** The model adapts to the task extremely quickly by leveraging its existing knowledge of English.
3.  **Hugging Face:** The `Trainer` and `pipeline` APIs simplify complex operations, making Transformer fine-tuning accessible and robust.

This concludes our fine-tuning demonstration. Next, we will explore **Self-Supervised Learning**, the technique used to create the pre-trained weights in the first place.